# Loading Data

Dataset Link: https://www.kaggle.com/datasets/muratkokludataset/rice-image-dataset

In [ ]:
# Download the public dataset without storing credentials in this notebook
import os
import glob
import kagglehub

dataset_path = kagglehub.dataset_download("muratkokludataset/rice-image-dataset")
matches = glob.glob(os.path.join(dataset_path, "**", "Rice_Image_Dataset"), recursive=True)
if not matches:
    raise FileNotFoundError("Rice_Image_Dataset directory was not found after download.")
data_dir = matches[0]
print(f"Dataset directory: {data_dir}")


In [ ]:
# Verify the downloaded dataset and count images per class
print("Exists?", os.path.isdir(data_dir))
print("Class folders:", os.listdir(data_dir))

counts = {}
for cls in sorted(os.listdir(data_dir)):
    class_dir = os.path.join(data_dir, cls)
    if os.path.isdir(class_dir):
        n = len(glob.glob(os.path.join(class_dir, "*.jpg")))
        print(f"{cls}: {n}")
        counts[cls] = n


# Exploratory Data Analysis (EDA)

In [ ]:
# =====================================================
# EDA — Bar Plot of Image Count per Class
# Purpose: Visualize class balance/imbalance
# =====================================================
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
# Exclude the .txt file from plotting
plot_counts = {k: v for k, v in counts.items() if k.endswith('.txt') is False}
plt.bar(list(plot_counts.keys()), list(plot_counts.values()))
plt.title("Image Count per Class")
plt.xlabel("Rice Variety")
plt.ylabel("Number of Images")
plt.xticks(rotation=30)
plt.show()

In [ ]:
# =====================================================
# Show 10 Random Images per Class WITH Labels on Each Tile
# Purpose: Quick visual inspection of each class
# =====================================================
import random, matplotlib.image as mpimg
import matplotlib.pyplot as plt
import os

COLS = 10
classes = [k for k in counts.keys() if k.endswith('.txt') is False] # Define classes
ROWS = len(classes)

plt.figure(figsize=(COLS*2, ROWS*2))

for i, cls in enumerate(classes):
    folder = os.path.join(data_dir, cls) # Use data_dir
    files  = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))]
    sample_files = random.sample(files, k=min(COLS, len(files)))
    if len(sample_files) < COLS and len(files) >= COLS:
        sample_files = files[:COLS]

    for j, fname in enumerate(sample_files[:COLS]):
        path = os.path.join(folder, fname)
        img = mpimg.imread(path)
        ax = plt.subplot(ROWS, COLS, i*COLS + j + 1)
        ax.imshow(img)
        ax.axis("off")
        # ✅ Add label for every tile
        ax.set_title(cls, fontsize=9)

plt.suptitle("Ten Random Images per Class", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Check Image Shapes and Channels
import os
import cv2
from collections import Counter

image_shapes = []

for cls in classes:
    folder = os.path.join(data_dir, cls)
    files = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    for f in files[:100]:  # Only check first 100 per class to save time
        img_path = os.path.join(folder, f)
        img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)  # Keep all channels
        if img is not None:
            image_shapes.append(img.shape)

# Count unique shapes
shape_counts = Counter(image_shapes)
print("Unique image shapes and their counts:")
for shape, count in shape_counts.items():
    print(f"{shape}: {count}")


In [ ]:
# brightness distribution of images
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
from collections import Counter


brightness_per_class = {}

for cls in classes:
    folder = os.path.join(data_dir, cls)
    files = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    brightness_values = []

    for f in files[:100]:
        img = cv2.imread(os.path.join(folder, f))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        brightness_values.append(np.mean(gray))

    brightness_per_class[cls] = brightness_values

# Plot
plt.figure(figsize=(10, 6))
for cls in brightness_per_class:
    plt.hist(brightness_per_class[cls], bins=20, alpha=0.5, label=cls)

plt.title("Brightness Distribution per Class")
plt.xlabel("Average Pixel Intensity")
plt.ylabel("Image Count")
plt.legend()
plt.show()

In [ ]:
# Pixel Intensity Histogram
import matplotlib.pyplot as plt
import numpy as np

all_pixels = []

for cls in classes:
    folder = os.path.join(data_dir, cls)
    files = os.listdir(folder)[:100]  # sample first 100 images
    for f in files:
        img_path = os.path.join(folder, f)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            all_pixels.extend(img.flatten())

plt.figure(figsize=(8, 4))
plt.hist(all_pixels, bins=50, color='gray')
plt.title("Histogram of Pixel Intensities (All Images)")
plt.xlabel("Pixel Value (0-255)")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

In [ ]:
# Pixel Intensity Histograms Per Class
plt.figure(figsize=(12, 8))
for i, cls in enumerate(classes):
    cls_pixels = []
    folder = os.path.join(data_dir, cls)
    files = os.listdir(folder)[:100]
    for f in files:
        img_path = os.path.join(folder, f)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            cls_pixels.extend(img.flatten())

    plt.subplot(2, 3, i+1)
    plt.hist(cls_pixels, bins=50, color='steelblue')
    plt.title(cls)
    plt.xlabel("Pixel Intensity")
    plt.ylabel("Count")
    plt.tight_layout()

plt.suptitle("Pixel Intensity Distributions per Class", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================
# RGB histograms — one random image per class
# =====================================================
import os, random, cv2
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 8))
for i, cls in enumerate(classes):
    folder = os.path.join(data_dir, cls)
    files  = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))]
    img_bgr = cv2.imread(os.path.join(folder, random.choice(files)))

    plt.subplot(2, 3, i+1)      # 5 classes fits in 2x3 grid
    for ch_idx, ch_name, col in [(0,'Blue','b'), (1,'Green','g'), (2,'Red','r')]:
        hist = cv2.calcHist([img_bgr], [ch_idx], None, [256], [0,256]).ravel()
        plt.plot(hist, col, label=ch_name)
    plt.title(f"{cls}")
    plt.xlim([0,256])
    if i == 0: plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Mean and Std Dev of Pixels per Class

stats = {}

for cls in classes:
    pixels = []
    folder = os.path.join(data_dir, cls)
    files = os.listdir(folder)[:100]
    for f in files:
        img_path = os.path.join(folder, f)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            pixels.extend(img.flatten())

    pixels = np.array(pixels)
    stats[cls] = {
        "mean": np.mean(pixels),
        "std": np.std(pixels)
    }

print("Pixel Intensity Stats per Class:")
for cls, s in stats.items():
    print(f"{cls}: Mean = {s['mean']:.2f}, Std = {s['std']:.2f}")

In [ ]:
# Outlier Detection via Blurriness
import cv2
import os
import matplotlib.pyplot as plt

# Blurriness threshold (empirical value)
BLURRY_THRESHOLD = 50.0

# Store results
blurry_stats = {}
blurry_examples = {}

# Loop through each class folder
for cls in classes:
    folder = os.path.join(data_dir, cls)
    blurry_count = 0
    blurry_images = []

    files = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.png', '.jpeg', '.bmp'))]

    for f in files[:200]:  # limit to 200 for faster demo
        path = os.path.join(folder, f)
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

        if img is not None:
            # Calculate Laplacian Variance
            score = cv2.Laplacian(img, cv2.CV_64F).var()

            if score < BLURRY_THRESHOLD:
                blurry_count += 1
                blurry_images.append((f, score))

    blurry_stats[cls] = blurry_count
    blurry_examples[cls] = blurry_images[:3]  # keep top 3 blurry examples per class

# Print result
print("\n🔍 Blurriness Report (using Laplacian Variance < 50):")
for cls in blurry_stats:
    print(f" {cls}: {blurry_stats[cls]} blurry images (out of ~200)")

# Optional: Plot a few blurry examples
plt.figure(figsize=(12, 8))
i = 1
for cls in blurry_examples:
    for fname, score in blurry_examples[cls]:
        img_path = os.path.join(data_dir, cls, fname)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        plt.subplot(len(blurry_examples), 3, i)
        plt.imshow(img, cmap='gray')
        plt.title(f"{cls}\nBlur Score: {score:.1f}", fontsize=9)
        plt.axis('off')
        i += 1

plt.suptitle("Examples of Blurry Images Detected (Laplacian Variance)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Mean and Std Dev of Pixels per Class and Blurry Count
import pandas as pd
summary_df = pd.DataFrame({
    'Mean': [stats[c]['mean'] for c in classes],
    'StdDev': [stats[c]['std'] for c in classes],
    'Blurry Count': [blurry_stats[c] for c in classes],
    'Sample Count': [counts[c] for c in classes]
}, index=classes)
print(summary_df)

# Preprocessing

In [ ]:
# Step 1: Load Images and Labels into Arrays
import os
import numpy as np
import cv2

# Store image data and labels
X = []
y = []

label_map = {cls: idx for idx, cls in enumerate(classes)}  # class to integer

# Loop through each class folder and load images
for cls in classes:
    folder = os.path.join(data_dir, cls)
    files = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.png', '.jpeg', '.bmp'))]

    for fname in files:
        img_path = os.path.join(folder, fname)

        # Read image as grayscale
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if img is not None:
            # Resize to 75×75
            img = cv2.resize(img, (75, 75))

            # Convert to array and add channel dimension → (75, 75, 1)
            img = img.reshape(75, 75, 1)

            X.append(img)
            y.append(label_map[cls])  # encode label

In [ ]:
# Step 2: Convert to NumPy Arrays & Normalize (0–1)
X = np.array(X, dtype='float32')
y = np.array(y)

# Normalize pixel values to 0–1
X = X / 255.0

In [ ]:
# Step 3: One-Hot Encode Labels
from tensorflow.keras.utils import to_categorical

# One-hot encode for softmax classification
y_cat = to_categorical(y, num_classes=len(classes))

In [ ]:
y_cat

In [ ]:
# Step 4: Stratified Train–Val–Test Split (70/15/15)
from sklearn.model_selection import train_test_split

# First split: Train and temp (val+test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_cat, stratify=y, test_size=0.30, random_state=42
)

# Second split: Val and test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, stratify=np.argmax(y_temp, axis=1), test_size=0.50, random_state=42
)

print("✅ Shapes:")
print("Train:", X_train.shape)
print("Val:  ", X_val.shape)
print("Test: ", X_test.shape)

In [ ]:
# Data Augmentation Pipeline (Train Only)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define augmentation generator
train_datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

# For validation/test: only rescale (no augmentation)
val_test_datagen = ImageDataGenerator()


# train_datagen.fit(X_train)

# Create generators
train_generator = train_datagen.flow(X_train, y_train, batch_size=32, shuffle=True)
val_generator   = val_test_datagen.flow(X_val, y_val, batch_size=32, shuffle=False)
test_generator  = val_test_datagen.flow(X_test, y_test, batch_size=32, shuffle=False)

ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
)

# Custom CNN

## Baseline CNN for Rice Image Classification

In [ ]:
# =====================================================
# 🧪 Step 1: Import Libraries
# =====================================================
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# =====================================================
# 🏗️ Step 2: Build the Baseline CNN Model
# - Uses 3+ hidden layers: Conv → Pool → Dense
# =====================================================
model = Sequential()

# 🧱 Hidden Layer 1: Convolution + ReLU + MaxPooling
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(75, 75, 1)))
model.add(MaxPooling2D(pool_size=(2, 2)))

# 🧱 Hidden Layer 2: Convolution + ReLU + MaxPooling
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# 🧱 Hidden Layer 3: Convolution + ReLU
model.add(Conv2D(128, (3, 3), activation='relu'))

# 🧱 Flatten layer to go from 3D to 1D
model.add(Flatten())

# 🧠 Dense Layer: Fully connected
model.add(Dense(128, activation='relu'))

# 🔚 Output Layer: 5 rice classes (softmax for multi-class)
model.add(Dense(5, activation='softmax'))

In [ ]:
# =====================================================
# ⚙️ Step 3: Compile the Model
# - Loss: categorical_crossentropy (multi-class)
# - Optimizer: Adam with learning rate 0.001
# =====================================================
model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

# =====================================================
# Step 4: Summary of the Model
# =====================================================
model.summary()

In [ ]:
# 🚀 Train the Baseline CNN
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    verbose=1)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))  # Width x Height

# 📈 Plot Accuracy
plt.subplot(1, 2, 1)  # (rows, columns, position)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# 📉 Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# 🖼️ Show both plots
plt.tight_layout()
plt.show()

## Evaluate the model on the test set

In [ ]:
# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(test_generator, verbose=1)
print(f"✅ Test Accuracy: {test_acc:.4f}")
print(f"✅ Test Loss: {test_loss:.4f}")

## Confusion Matrix and Classification Report

In [ ]:
# Confusion Matrix and Classification Report
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Predict class probabilities
y_pred_probs = model.predict(test_generator)

# Get predicted labels (argmax)
y_pred_labels = np.argmax(y_pred_probs, axis=1)

# Get true labels (convert one-hot to class index)
y_true_labels = np.argmax(y_test, axis=1)

# ✅ Classification Report
print(classification_report(y_true_labels, y_pred_labels, target_names=classes))

# ✅ Confusion Matrix
cm = confusion_matrix(y_true_labels, y_pred_labels)

# ✅ Plot the Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title("Baseline CNN - Confusion Matrix")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.tight_layout()
plt.show()

## Regularized CNN

In [ ]:
# =====================================================
# 🧪 Step 1: Import Libraries
# =====================================================
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2
# =====================================================
# 🏗️ Step 2: Build Regularized CNN
# - Same structure as baseline
# - Add Dropout after key layers
# =====================================================
model = Sequential()

# 🧱 Conv Layer 1 + BN + MaxPooling + Dropout
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(75, 75, 1)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

# 🧱 Conv Layer 2 + BN + MaxPooling + Dropout
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

# 🧱 Conv Layer 3 + BN
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.3))   # slight dropout after deeper conv

# 🧱 Flatten
model.add(Flatten())

# 🧠 Dense Layer + BN + Dropout + L2
model.add(Dense(128, activation='relu', kernel_regularizer=l2(0.001)))
model.add(BatchNormalization())
model.add(Dropout(0.4))

# 🔚 Output Layer (5 classes)
model.add(Dense(5, activation='softmax'))

In [ ]:
# =====================================================
# ⚙️ Step 3: Compile the Model
# - Optimizer: Adam with learning rate 0.001
# - Loss: categorical_crossentropy (since using one-hot labels)
# - Metric: accuracy
# =====================================================
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)# =====================================================
# 📌 Step 4: Add EarlyStopping Callback
# - Monitor: val_loss
# - Patience: 5 epochs (wait for improvement)
# - Restore best weights to avoid overfitting
# - Verbose: 1 (so you can see when it stops early)
# =====================================================
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)
# =====================================================
# Summary of the Model
# =====================================================
model.summary()

In [ ]:
# =====================================================
# 🚀 Step 5: Train the Model
# =====================================================
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

# 📈 Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Regularized CNN - Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# 📉 Loss plot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Regularized CNN - Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Evaluate the model on the test set

In [ ]:
# ✅ Evaluate the model on the test set
test_loss, test_acc = model.evaluate(test_generator, verbose=1)
print(f"✅ Test Accuracy: {test_acc:.4f}")
print(f"✅ Test Loss: {test_loss:.4f}")

## Classification Report & Confusion Matrix

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Get predictions (probabilities)
y_pred_probs = model.predict(test_generator)

# Convert probabilities → predicted class indices
y_pred_labels = np.argmax(y_pred_probs, axis=1)

# True labels (convert from one-hot encoding to class indices)
y_true_labels = np.argmax(y_test, axis=1)

# ✅ Classification Report
print("\n📊 Classification Report:")
print(classification_report(y_true_labels, y_pred_labels, target_names=classes))

# ✅ Confusion Matrix
cm = confusion_matrix(y_true_labels, y_pred_labels)

# ✅ Plot Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title("Regularized CNN - Confusion Matrix")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.tight_layout()
plt.show()

## Manual Tuning

In [ ]:
# =====================================================
# 🧪 Step 1: Import Required Libraries
# =====================================================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras import backend as K

In [ ]:
# =====================================================
# 🏗️ Step 2: Define the Function to Manually Tune the Model (Updated)
# - Uses stronger regularization, larger Dense layer, and slightly deeper dropout
# =====================================================

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Dropout, GlobalAveragePooling2D, Dense
from tensorflow.keras.regularizers import l2

def build_light_deep_model(input_shape=(75, 75, 1), num_classes=5, reg=0.001):

    model = Sequential()
    # Block 1
    model.add(Conv2D(16, (3, 3), activation='relu', input_shape=input_shape))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.2))
    # Block 2
    model.add(Conv2D(32, (3, 3), activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.2))
    # Block 3
    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))
    # Block 4 (optional deeper layer)
    model.add(Conv2D(128, (3, 3), activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))
    # 🧱 Flatten + Dense
    model.add(Flatten())
    model.add(Dense(128, activation='relu', kernel_regularizer=l2(reg)))
    model.add(BatchNormalization())
    model.add(Dropout(0.4))
    # Optional small dense layer for control
    model.add(Dense(64, activation='relu', kernel_regularizer=l2(reg)))
    model.add(Dropout(0.3))
    # Output
    model.add(Dense(num_classes, activation='softmax'))
    return model

In [ ]:
# =====================================================
# 📦 Step 3: Define Separate Data Generators (Manual Tuning)
# =====================================================

# Define augmentation generator
train_datagen_manual = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

# For validation: only rescale (no augmentation)
val_test_datagen_manual = ImageDataGenerator()

# Create generators with a batch size of 16
train_generator_manual = train_datagen_manual.flow(X_train, y_train, batch_size=16, shuffle=True)
val_generator_manual   = val_test_datagen_manual.flow(X_val, y_val, batch_size=16, shuffle=False)

print("✅ Separate data generators created for manual tuning with batch size 16.")

In [ ]:
# =====================================================
# ⚙️ Step 4: Compile the Model (Manual Tuning)
# =====================================================

# Build the model using the defined function (with default or specified hyperparameters)
# Example: Using default hyperparameters (lr=0.001, reg=0.001, dense_units=128)
model_manual = build_light_deep_model()

# Compile the model
model_manual.compile(
    optimizer=Adam(learning_rate=0.0001), # Example learning rate
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Model compiled for manual tuning.")

In [ ]:
# =====================================================
# ⏹️ Step 5: Add EarlyStopping to Stop Training if No Improvement (Manual Tuning)
# =====================================================
from tensorflow.keras.callbacks import EarlyStopping

early_stop_manual = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

print("✅ EarlyStopping callback defined for manual tuning.")

# =====================================================
# Summary of the Model
# =====================================================
model_manual.summary()

In [ ]:
# =====================================================
# 🚀 Step 6: Train the Model (Manual Tuning)
# =====================================================
history_manual = model_manual.fit(
    train_generator_manual,
    validation_data=val_generator_manual,
    epochs=20,
    callbacks=[early_stop_manual],
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

# Extract metrics from history
acc = history_manual.history['accuracy']
val_acc = history_manual.history['val_accuracy']
loss = history_manual.history['loss']
val_loss = history_manual.history['val_loss']

epochs = range(1, len(acc) + 1)

# Plot accuracy
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, acc, label='Train Accuracy')
plt.plot(epochs, val_acc, label='Validation Accuracy')
plt.title('Manual Tuning CNN - Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot loss
plt.subplot(1, 2, 2)
plt.plot(epochs, loss, label='Train Loss')
plt.plot(epochs, val_loss, label='Validation Loss')
plt.title('Manual Tuning CNN - Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Evaluate the Manual Tuning CNN on the test set

In [ ]:
# Evaluate the manual tuning model on the test set
test_loss_manual, test_acc_manual = model_manual.evaluate(val_test_datagen_manual.flow(X_test, y_test, batch_size=16, shuffle=False), verbose=1)
print(f"✅ Manual Tuning CNN Test Accuracy: {test_acc_manual:.4f}")
print(f"✅ Manual Tuning CNN Test Loss: {test_loss_manual:.4f}")

## Manual Tuning CNN Confusion Matrix and Classification Report

In [ ]:
# Confusion Matrix and Classification Report for Manual Tuning CNN
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Predict class probabilities
y_pred_probs_manual = model_manual.predict(val_test_datagen_manual.flow(X_test, y_test, batch_size=16, shuffle=False), verbose=1)

# Get predicted labels (argmax)
y_pred_labels_manual = np.argmax(y_pred_probs_manual, axis=1)

# Get true labels (convert one-hot to class index)
y_true_labels_manual = np.argmax(y_test, axis=1)

# ✅ Classification Report
print("\n📊 Manual Tuning CNN Classification Report:")
print(classification_report(y_true_labels_manual, y_pred_labels_manual, target_names=classes))

# ✅ Confusion Matrix
cm_manual = confusion_matrix(y_true_labels_manual, y_pred_labels_manual)

# ✅ Plot the Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm_manual, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title("Manual Tuning CNN - Confusion Matrix")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.tight_layout()
plt.show()

## Hyperparameter Tuning (Hyperband tuner Keras)

In [ ]:
# =====================================================
# 🧪 Step 1: Imports
# =====================================================
!pip install keras-tuner --quiet
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [ ]:
# =====================================================
# 🧠 Step 2: Define Model Builder
# =====================================================
def build_model(hp):
    model = Sequential()
    # 1st Conv Layer
    model.add(Conv2D(
        filters=hp.Choice('conv_1_filters', [32, 64]),
        kernel_size=(3, 3),
        activation='relu',
        input_shape=(75, 75, 1)
    ))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    # 2nd Conv Layer
    model.add(Conv2D(
        filters=hp.Choice('conv_2_filters', [64, 128]),
        kernel_size=(3, 3),
        activation='relu'
    ))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    # 3rd Conv Layer
    model.add(Conv2D(
        filters=hp.Choice('conv_3_filters', [128, 256]),
        kernel_size=(3, 3),
        activation='relu'
    ))
    model.add(Flatten())
    # Dense Layer
    model.add(Dense(
        units=hp.Choice('dense_units', [256, 512, 1024]),
        activation='relu'
    ))
    # Dropout
    model.add(Dropout(
        rate=hp.Choice('dropout_rate', [0.3, 0.4, 0.5])
    ))
    # Output
    model.add(Dense(5, activation='softmax'))
    # Compile
    model.compile(
        loss='categorical_crossentropy',
        optimizer=Adam(
            learning_rate=hp.Choice('lr', [1e-2, 1e-3, 1e-4])
        ),
        metrics=['accuracy']
    )
    return model

In [ ]:
# =====================================================
# 🧪 Step 3: Define Tuner
# =====================================================
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=25,
    factor=3,
    directory='kt_runs',
    project_name='final_cnn_tuning',
    overwrite=True,
    seed=42
)

In [ ]:
# =====================================================
# 🧪 Step 4: Define Callbacks
# =====================================================
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)

In [ ]:
# =====================================================
# 🚀 Step 5: Start Search
# =====================================================
tuner.search(
    train_generator,
    validation_data=val_generator,
    epochs=25,
    callbacks=[early_stop, reduce_lr]
)

## Get the Best Model

In [ ]:
# Get the best model
best_model = tuner.get_best_models(num_models=1)[0]
best_hps = tuner.get_best_hyperparameters(1)[0]

print("✅ Best Hyperparameters:")
print(f"Conv 1 Filters: {best_hps.get('conv_1_filters')}")
print(f"Conv 2 Filters: {best_hps.get('conv_2_filters')}")
print(f"Conv 3 Filters: {best_hps.get('conv_3_filters')}")
print(f"Dense Units: {best_hps.get('dense_units')}")
print(f"Dropout Rate: {best_hps.get('dropout_rate')}")
print(f"Learning Rate: {best_hps.get('lr')}")


# Print model summary
best_model.summary()

## Retrain best model on full train/val

In [ ]:
# Retrain best model on full train/val
# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, verbose=1)

# Train the model
history = best_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
# accuracy & loss
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))

# Accuracy (left)
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Model Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True)

# Loss (right)
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(True)

plt.tight_layout()
plt.show()

## Evaluate on test set

In [ ]:
test_loss, test_acc = best_model.evaluate(test_generator, verbose=1)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

## Classification Report & Confusion Matrix

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt # Import matplotlib

# Predict
y_pred = best_model.predict(test_generator)
y_pred_labels = np.argmax(y_pred, axis=1)

# Get true labels from the original y_test array
y_true = np.argmax(y_test, axis=1) # Use y_test and convert from one-hot to labels

# Classification report
print(classification_report(y_true, y_pred_labels, target_names=classes)) # Add target_names

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes) # Add labels to heatmap
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## Rebuild Best CNN model after tuning

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Set best hyperparameters
conv1_filters = 64
conv2_filters = 64
conv3_filters = 128
dense_units = 512
dropout_rate = 0.3
learning_rate = 0.001
input_shape = (75, 75, 1)

# Build the Best model
best_model = Sequential([
    Conv2D(conv1_filters, kernel_size=(3, 3), activation='relu', input_shape=input_shape),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(conv2_filters, kernel_size=(3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(conv3_filters, kernel_size=(3, 3), activation='relu'),

    Flatten(),
    Dense(dense_units, activation='relu'),
    Dropout(dropout_rate),
    Dense(5, activation='softmax')
])

# Compile the model
optimizer = Adam(learning_rate=learning_rate)
best_model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Show model summary
best_model.summary()


## Retrain the Model

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, verbose=1)

history = best_model.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=10,
                    batch_size=32,
                    callbacks=[early_stop, reduce_lr])

## Plot Accuracy

In [ ]:
# accuracy & loss
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))

# Accuracy (left)
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Model Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True)

# Loss (right)
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(True)

plt.tight_layout()
plt.show()

## Evaluation and Confusion Matrix and Classification Repor

In [ ]:
# 🧪 1. Evaluate on Test Set
test_loss, test_acc = best_model.evaluate(test_generator, verbose=1)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Predict class probabilities
y_pred_probs = best_model.predict(test_generator, verbose=0)

# Get predicted class labels
y_pred = np.argmax(y_pred_probs, axis=1)

# Get true class labels from the test_generator
# This requires iterating through the generator to get all true labels
y_true = []
for i in range(len(test_generator)):
    batch_labels = test_generator[i][1] # Get labels from the batch
    y_true.extend(np.argmax(batch_labels, axis=1)) # Convert one-hot to labels

# Ensure y_true and y_pred have the same length
y_true = y_true[:len(y_pred)]

# Classification report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=classes))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

# Plot the Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# Save the best model
best_model.save('best_cnn_model.h5')

In [ ]:
!pip install visualkeras -q

In [ ]:
import visualkeras
from tensorflow.keras.models import load_model
from IPython.display import Image

# Create the architecture diagram
visualkeras.layered_view(best_model, to_file="best_model_architecture.png", legend=True) # Use best_model

# Display the image in notebook
Image(filename="best_model_architecture.png")

## Predict a Random Sample from the Test Set

In [ ]:
import random

# Pick random index
idx = random.randint(0, len(X_test)-1)

# Select the image and true label
sample_img = X_test[idx]
true_label = np.argmax(y_test[idx])  # one-hot → class index

# Reshape for prediction
input_img = sample_img.reshape(1, 75, 75, 1)
pred_probs = model.predict(input_img)
pred_class = np.argmax(pred_probs, axis=1)[0]

# Map index to class name
label_map = {i: cls for i, cls in enumerate(classes)}

print(f"✅ True Label: {label_map[true_label]}")
print(f"🤖 Predicted: {label_map[pred_class]}")

# Plot the image
plt.imshow(sample_img.squeeze(), cmap='gray')
plt.title(f"True: {label_map[true_label]} | Pred: {label_map[pred_class]}")
plt.axis("off")
plt.show()

## CNN Predictions on 10 Random Test Samples

In [ ]:
import random
import matplotlib.pyplot as plt

# Pick 10 random indexes from the test set
indices = random.sample(range(len(X_test)), 10)

plt.figure(figsize=(15, 8))

for i, idx in enumerate(indices):
    # Get sample image and true label
    sample_img = X_test[idx]
    true_label = np.argmax(y_test[idx])

    # Preprocess for prediction
    input_img = sample_img.reshape(1, 75, 75, 1)
    pred_probs = model.predict(input_img, verbose=0)
    pred_class = np.argmax(pred_probs, axis=1)[0]

    # Plot image
    plt.subplot(2, 5, i+1)  # 2 rows, 5 columns
    plt.imshow(sample_img.squeeze(), cmap='gray')
    plt.axis("off")

    # Title with true and predicted labels
    color = "green" if true_label == pred_class else "red"
    plt.title(f"T: {classes[true_label]}\nP: {classes[pred_class]}", color=color, fontsize=10)

plt.suptitle("CNN Predictions on 10 Random Test Samples", fontsize=14)
plt.tight_layout()
plt.show()

## CNN Predictions on 10 Random Test Samples with Predicted and Original

In [ ]:
import random
import matplotlib.pyplot as plt
import cv2, os

# Pick 10 random indexes from the test set
indices = random.sample(range(len(X_test)), 10)

plt.figure(figsize=(20, 8))  # Wide figure for 2 rows × 10 samples

for i, idx in enumerate(indices):
    # Original grayscale from your dataset (already 75x75 preprocessed)
    sample_img = X_test[idx]
    true_label = np.argmax(y_test[idx])

    # Predict
    input_img = sample_img.reshape(1, 75, 75, 1)
    pred_probs = model.predict(input_img, verbose=0)
    pred_class = np.argmax(pred_probs, axis=1)[0]

    # ---- Show the original high-resolution image ----
    true_class_name = classes[true_label]
    folder = os.path.join(data_dir, true_class_name)
    orig_file = os.listdir(folder)[0]  # just grab one image from the folder
    orig_img = cv2.imread(os.path.join(folder, orig_file))
    orig_img = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)

    # Each sample takes up 2 columns → so total columns = 10
    plt.subplot(2, 10, 2*i+1)  # left = Original
    plt.imshow(orig_img)
    plt.axis("off")
    plt.title(f"Original\n({true_class_name})", fontsize=8)

    plt.subplot(2, 10, 2*i+2)  # right = Resized CNN input
    plt.imshow(sample_img.squeeze(), cmap="gray")
    color = "green" if true_label == pred_class else "red"
    plt.title(f"T: {classes[true_label]}\nP: {classes[pred_class]}", color=color, fontsize=8)
    plt.axis("off")

plt.suptitle("Original vs CNN Input (10 Random Test Samples)", fontsize=14)
plt.tight_layout()
plt.show()

# Pre-Trained Model(EfficientNetB0)

## Build tf.data pipelines

In [ ]:
# =====================================================
# 🧪 Step 1: Make tf.data pipelines (RGB + resize + preprocess)
# =====================================================
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

IMG_SIZE = (224, 224)
BATCH = 32

def prep_for_efficientnet(x, y):
    x = tf.image.grayscale_to_rgb(x)
    x = tf.image.resize(x, IMG_SIZE)
    x = preprocess_input(x * 255.0)
    return x, y

AUTOTUNE = tf.data.AUTOTUNE

def make_ds(x, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    ds = ds.map(prep_for_efficientnet, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(buffer_size=1000)
    ds = ds.batch(32).prefetch(AUTOTUNE)
    return ds

# splits
train_ds = make_ds(X_train, y_train, training=True)
val_ds   = make_ds(X_val,   y_val)
test_ds  = make_ds(X_test,  y_test)

## Build the transfer-learning model (freeze base)

In [ ]:
# =====================================================
# 🏗️ Step 2: Build EfficientNetB0 base (frozen) + head
# =====================================================
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.optimizers import Adam

# Data augmentation (lightweight, runs on GPU)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name="aug")

# Pretrained backbone
base = EfficientNetB0(include_top=False, weights="imagenet",
                      input_shape=(224,224,3))
base.trainable = False  # freeze

inputs = layers.Input(shape=(224,224,3))
x = data_augmentation(inputs)
x = base(x, training=False)             # important when frozen
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)              # regularization head
outputs = layers.Dense(5, activation="softmax")(x)

tl_model = models.Model(inputs, outputs, name="EffB0_transfer")

tl_model.compile(
    optimizer=Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

tl_model.summary()


## Train the frozen base (feature extractor)

In [ ]:
# =====================================================
# 🚀 Step 3: Train with frozen base
# =====================================================
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1)
rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1)

history_tl = tl_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early, rlrop],
    verbose=1
)

## Plot Accuracy and Loss of the Frozen Base

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

# 📈 Accuracy Plot (Left)
plt.subplot(1, 2, 1)
plt.plot(history_tl.history['accuracy'], label='Train Accuracy')
plt.plot(history_tl.history['val_accuracy'], label='Validation Accuracy')
plt.title('EfficientNetB0 - Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# 📉 Loss Plot (Right)
plt.subplot(1, 2, 2)
plt.plot(history_tl.history['loss'], label='Train Loss')
plt.plot(history_tl.history['val_loss'], label='Validation Loss')
plt.title('EfficientNetB0 - Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Evaluate the model on the test set

In [ ]:
# ✅ Evaluate the model on the test set
test_loss, test_accuracy = tl_model.evaluate(test_ds, verbose=1)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

## Confusion Matrix and Classification Report

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Get true labels and predictions for the frozen model (tl_model)
y_true_frozen = []
y_pred_frozen = []

for images, labels in test_ds:
    # Use the tl_model before fine-tuning
    preds = tl_model.predict(images, verbose=0)
    y_true_frozen.extend(np.argmax(labels.numpy(), axis=-1))
    y_pred_frozen.extend(np.argmax(preds, axis=-1))

# Generate the confusion matrix
cm_frozen = confusion_matrix(y_true_frozen, y_pred_frozen)

# Plot it with class names
classes = ['Arborio', 'Basmati', 'Ipsala', 'Jasmine', 'Karacadag'] # Define classes
plt.figure(figsize=(10, 8)) # Slightly larger figure for better readability
sns.heatmap(cm_frozen, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('EfficientNetB0 (Frozen) – Confusion Matrix (Test Set)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

print("Classification Report:")
print(classification_report(y_true_frozen, y_pred_frozen, target_names=classes, digits=4)) # Add target_names

## Fine-tune: unfreeze top layers of the backbone

In [ ]:
# ✅ Unfreeze top 30 layers only
for layer in base.layers[:-30]:
    layer.trainable = False
for layer in base.layers[-30:]:
    layer.trainable = True

# ✅ Compile with lower LR
tl_model.compile(
    optimizer=Adam(1e-5),  # recommended fine-tuning LR
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ✅ Retrain for few epochs with callbacks
history_ft = tl_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early, rlrop],
    verbose=1
)

## Plot Accuracy and Loss of Fine Tuning

In [ ]:
# Plot Accuracy

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

# 📈 Accuracy Plot
plt.subplot(1, 2, 1)
plt.plot(history_ft.history['accuracy'], label='Train Accuracy')
plt.plot(history_ft.history['val_accuracy'], label='Validation Accuracy')
plt.title('Fine-Tuning Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# 📉 Loss Plot
plt.subplot(1, 2, 2)
plt.plot(history_ft.history['loss'], label='Train Loss')
plt.plot(history_ft.history['val_loss'], label='Validation Loss')
plt.title('Fine-Tuning Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Evaluate the performance of the model on the test set

In [ ]:
# ✅ Evaluate the performance of the model on the test set
test_loss, test_accuracy = tl_model.evaluate(test_ds, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

## Confusion Matrix and Classification Report of Fine Tuning

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Get true labels and predictions
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = tl_model.predict(images, verbose=0) # Use tl_model
    y_true.extend(np.argmax(labels.numpy(), axis=-1))
    y_pred.extend(np.argmax(preds, axis=-1))

# Generate the confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Plot it with class names
classes = ['Arborio', 'Basmati', 'Ipsala', 'Jasmine', 'Karacadag'] # Define classes
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('EfficientNetB0 (Fine-Tuning) – Confusion Matrix (Test Set)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=classes, digits=4)) # Add target_names

## Hyperparameter Tuning for Transfer Learning

In [ ]:
!pip install -q keras-tuner

In [ ]:
# =====================================================
# 🧪 Step A: Hyperparameter Tuning for EfficientNetB0
# - Tune: dense_units, dropout, learning rate, and how many top layers to unfreeze
# =====================================================
import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import EfficientNetB0

IMG_SIZE = (224, 224)

def build_tl_model(hp):
    # Data augmentation (kept light)
    aug = tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
    ], name="aug")

    # Pretrained backbone
    base = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(224,224,3))
    base.trainable = False  # start frozen

    # Tunable head
    dense_units = hp.Choice("dense_units", [256, 512, 768])
    dropout_rate = hp.Choice("dropout", [0.3, 0.4, 0.5])
    lr = hp.Choice("lr", [1e-3, 5e-4, 1e-4])

    # Unfreeze top N layers from the start (small number to remain stable)
    unfreeze_top = hp.Choice("unfreeze_top_layers", [0, 10, 20, 30])
    if unfreeze_top > 0:
        for layer in base.layers[-unfreeze_top:]:
            layer.trainable = True

    inputs = layers.Input(shape=(224,224,3))
    x = aug(inputs)
    x = base(x, training=(unfreeze_top > 0))   # if any part is trainable, set training=True
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(5, activation="softmax")(x)

    model = models.Model(inputs, outputs, name="EffB0_Tuned")
    model.compile(optimizer=Adam(lr), loss="categorical_crossentropy", metrics=["accuracy"])
    return model

tuner = kt.Hyperband(
    build_tl_model,
    objective="val_accuracy",
    max_epochs=12,
    factor=3,
    directory="kt_tl_runs",
    project_name="effb0_transfer",
    overwrite=True,
    seed=42
)

early = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1)
rlrop = tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1)

tuner.search(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=[early, rlrop],
    verbose=1
)

tuner.results_summary()

## Get the Best Model

In [ ]:
# ============================
# 🎯 Best hyperparameters & model
# ============================
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
best_model = tuner.get_best_models(num_models=1)[0]

# ✅ Print best hyperparameter values
print("✅ Best Hyperparameters:")
print(f"Dense Units: {best_hps.get('dense_units')}")
print(f"Dropout Rate: {best_hps.get('dropout')}")
print(f"Learning Rate: {best_hps.get('lr')}")
print(f"Unfreeze Top Layers: {best_hps.get('unfreeze_top_layers')}")

# 🔎 Show model summary
best_model.summary()


## Retrain Best Model

In [ ]:
# # =====================================================
# 🧪 Step B: Retrain Best Model
# =====================================================
history = best_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early, rlrop],
    verbose=1
)

## Plot Accuracy and Loss

In [ ]:
import matplotlib.pyplot as plt

# Extract metrics from history
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(1, len(acc) + 1)

# Plot accuracy
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, acc, label='Train Accuracy')
plt.plot(epochs, val_acc, label='Validation Accuracy')
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot loss
plt.subplot(1, 2, 2)
plt.plot(epochs, loss, label='Train Loss')
plt.plot(epochs, val_loss, label='Validation Loss')
plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Evaluation on Test Set

In [ ]:
# =====================================================
# 🧪 Step C: Evaluation on Test Set
# =====================================================
test_loss, test_acc = best_model.evaluate(test_ds, verbose=1)
print(f"Test Accuracy: {test_acc:.4f}  |  Test Loss: {test_loss:.4f}")

## Confusion Matrix & Classification Report

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt # Import matplotlib

# Predict
y_pred_probs = best_model.predict(test_ds, verbose=0) # Predict using the best_model
y_pred_labels = np.argmax(y_pred_probs, axis=1)

# Get true labels from the test_ds
y_true = []
for _, labels in test_ds:
    y_true.extend(np.argmax(labels.numpy(), axis=-1))


# Classification report
classes = ['Arborio', 'Basmati', 'Ipsala', 'Jasmine', 'Karacadag']
print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred_labels, target_names=classes))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=45, ha='right') # Rotate labels for better readability
plt.yticks(rotation=0)
plt.tight_layout() # Adjust layout to prevent labels overlapping
plt.show()

## Rebuild EfficientNetB0 Tuned Model with Best Hyperparameters

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# 📌 Reuse the same input shape
INPUT_SHAPE = (224, 224, 3)

# 📌 Build the tuned model
def build_best_effb0_model():
    # Data Augmentation
    aug = Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.RandomZoom(0.1),
    ], name="aug")

    # Base model
    base = EfficientNetB0(include_top=False, weights="imagenet", input_shape=INPUT_SHAPE)

    # Unfreeze last 30 layers
    for layer in base.layers[-30:]:
        layer.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False
    # Build the model
    inputs = Input(shape=INPUT_SHAPE, name="input_layer")
    x = aug(inputs)
    x = base(x, training=True)
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    outputs = Dense(5, activation='softmax')(x)

    model = Model(inputs, outputs, name="EffB0_Tuned")

    # Compile the model
    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model
# ✅ Build the model
EffB0_Tuned = build_best_effb0_model()
EffB0_Tuned.summary()

## Train the Rebuild Best Model

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Define Callbacks
early = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1)
rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1)

history = EffB0_Tuned.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early, rlrop],
    verbose=1
)


In [ ]:
import matplotlib.pyplot as plt

# Extract metrics from history
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(1, len(acc) + 1)

# Plot accuracy
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, acc, label='Train Accuracy')
plt.plot(epochs, val_acc, label='Validation Accuracy')
plt.title('EfficientNetB0 Tuned_Best Model - Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot loss
plt.subplot(1, 2, 2)
plt.plot(epochs, loss, label='Train Loss')
plt.plot(epochs, val_loss, label='Validation Loss')
plt.title('EfficientNetB0 Tuned_Best Model - Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Evaluate test accuracy and loss

In [ ]:
# 📊 Evaluate test accuracy and loss
test_loss, test_acc = EffB0_Tuned.evaluate(test_ds, verbose=1)
print(f"✅ Test Accuracy: {test_acc:.4f}")
print(f"❌ Test Loss: {test_loss:.4f}")


## Classification Report & Confusion Matrix

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt # Import matplotlib

# 1️⃣ Get true labels and predicted labels
y_true = np.concatenate([y for x, y in test_ds], axis=0)
y_pred_probs = EffB0_Tuned.predict(test_ds, verbose=0) # Predict using the best_model
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_true, axis=1)

# Classification report
classes = ['Arborio', 'Basmati', 'Ipsala', 'Jasmine', 'Karacadag']
print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=classes))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Save the Pretrained (Tuned) Model

In [ ]:
# =====================================================
# 💾 Step F: Save the Pretrained (Tuned) Model
# =====================================================
EffB0_Tuned.save("efficientnetb0_transfer_tuned.h5")
print("Saved: efficientnetb0_transfer_tuned.h5")

In [ ]:
!pip install visualkeras -q

In [ ]:
import visualkeras
from tensorflow.keras.models import load_model
from IPython.display import Image

# Create the architecture diagram
visualkeras.layered_view(best_model, to_file="best_model_architecture.png", legend=True)

# Display the image in notebook
Image(filename="best_model_architecture.png")

## Show 1 image only EfficientNetB0 (Pretrained Model)

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf # Import tensorflow

# Convert test_ds into NumPy arrays for easier random sampling
# We need to convert the tf.data.Dataset back to numpy for simple random sampling
x_test_np = []
y_test_np = []

for images, labels in test_ds:
    x_test_np.append(images.numpy())
    y_test_np.append(labels.numpy())

x_test_np = np.concatenate(x_test_np, axis=0)
y_test_np = np.concatenate(y_test_np, axis=0)

# Create label map (index to class name) from the original 'classes' list
# classes = ['Arborio','Basmati','Ipsala','Jasmine','Karacadag'] # Uncomment and define if not available
label_map = {i: cls for i, cls in enumerate(classes)}

# Pick a random index
idx = random.randint(0, len(x_test_np) - 1)

# Select the image and true label (as index)
sample_img = x_test_np[idx]
true_label_idx = np.argmax(y_test_np[idx])

# Reshape and expand dims for prediction (EffNet expects RGB shape)
# The test_ds pipeline already resized and preprocessed the images, so we just need expand_dims
input_img = np.expand_dims(sample_img, axis=0)

# Predict using the best model
pred_probs = best_model.predict(input_img, verbose=0)
pred_idx = np.argmax(pred_probs, axis=1)[0]

# Map index to class name
true_label = label_map[true_label_idx]
pred_label = label_map[pred_idx]

# Print results
print(f"✅ True Label: {true_label}")
print(f"🤖 Predicted: {pred_label}")

# Plot the image
plt.imshow(sample_img.astype("uint8"))  # Convert to uint8 for correct display if needed
plt.title(f"True: {true_label} | Pred: {pred_label}",
          color='green' if true_label == pred_label else 'red')
plt.axis("off")
plt.show()

## Random Samples Prediction from Test Data

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

# ✅ Assume x_test_np, y_test_np, best_model, and classes are already defined
# If not, re-run conversion of test_ds into NumPy arrays

# Create label map (index to class name)
label_map = {i: cls for i, cls in enumerate(classes)}

# Step 1: Pick 9 random test samples
num_samples = 9
indices = random.sample(range(len(x_test_np)), num_samples)

# Step 2: Plot predictions
plt.figure(figsize=(9, 9))  # Smaller size for compact display

for i, idx in enumerate(indices):
    img = x_test_np[idx]
    true_label_idx = np.argmax(y_test_np[idx])
    true_label = label_map[true_label_idx]

    # Predict using pretrained model
    input_img = np.expand_dims(img, axis=0)
    pred_probs = best_model.predict(input_img, verbose=0)
    pred_idx = np.argmax(pred_probs)
    pred_label = label_map[pred_idx]
    confidence = np.max(pred_probs)

    # Plot
    plt.subplot(3, 3, i + 1)
    plt.imshow(img.astype("uint8"))
    plt.axis('off')
    color = 'green' if true_label == pred_label else 'red'
    plt.title(f"True: {true_label}\nPred: {pred_label} ({confidence:.1%})",
              fontsize=9, color=color)

plt.tight_layout()
plt.suptitle("EfficientNetB0 Predictions (9 Random Test Samples)", fontsize=12, y=1.03)
plt.show()

## Random Sample Testes From External Source Images

## Random Sample From Google Image

In [ ]:
from tensorflow.keras.utils import plot_model
from IPython.display import Image, display

# Generate the model architecture plot
plot_model(EffB0_Tuned, to_file='best_model_plot.png', show_shapes=True, show_layer_names=True)

# Display the saved image
display(Image(filename='best_model_plot.png'))